<a href="https://colab.research.google.com/github/mdsiamahmed26/coronary-artery/blob/main/coronary_artery_surface_roughness_measure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install VTK
!pip install vtk

import vtk
import numpy as np
import pandas as pd
from google.colab import files

# === Step 1. Upload your artery.vtp file ===
print("Please upload your .vtp file (artery surface)")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# === Step 2. Load lumen surface ===
reader = vtk.vtkXMLPolyDataReader()
reader.SetFileName(filename)
reader.Update()
surface = reader.GetOutput()

# === Step 3. Smooth copy of the surface ===
smooth = vtk.vtkWindowedSincPolyDataFilter()
smooth.SetInputData(surface)
smooth.SetNumberOfIterations(30)  # adjust smoothness
smooth.BoundarySmoothingOff()
smooth.Update()
smoothed_surface = smooth.GetOutput()

# === Step 4. Compute roughness values (distance between original & smooth) ===
dist = vtk.vtkDistancePolyDataFilter()
dist.SetInputData(0, surface)
dist.SetInputData(1, smoothed_surface)
dist.Update()

distance_array = dist.GetOutput().GetPointData().GetScalars()
values = np.array([distance_array.GetValue(i) for i in range(distance_array.GetNumberOfTuples())])
values_um = np.abs(values) * 1000  # mm → µm

# Attach roughness array to surface (for coloring in ParaView)
roughness_array = vtk.vtkFloatArray()
roughness_array.SetName("Roughness_um")
for v in values_um:
    roughness_array.InsertNextValue(v)
surface.GetPointData().AddArray(roughness_array)
surface.GetPointData().SetActiveScalars("Roughness_um")

# === Step 5. Compute metrics ===
Ra = np.mean(values_um)
Rq = np.sqrt(np.mean(values_um**2))
Rmax = np.max(values_um)
max_idx = np.argmax(values_um)
max_point = surface.GetPoint(max_idx)  # (x,y,z in mm)

print("=== Surface Roughness Results ===")
print(f"Ra: {Ra:.2f} µm")
print(f"Rq: {Rq:.2f} µm")
print(f"Rmax: {Rmax:.2f} µm")
print(f"Max roughness at point index {max_idx}, coordinates (mm): {max_point}")

# === Step 6. Save metrics to CSV ===
df = pd.DataFrame({
    "File": [filename],
    "Ra (µm)": [Ra],
    "Rq (µm)": [Rq],
    "Rmax (µm)": [Rmax],
    "MaxIndex": [max_idx],
    "X (mm)": [max_point[0]],
    "Y (mm)": [max_point[1]],
    "Z (mm)": [max_point[2]]
})
df.to_csv("roughness_results.csv", index=False)
files.download("roughness_results.csv")

# === Step 7. Create a point marker at the max roughness location ===
points = vtk.vtkPoints()
points.InsertNextPoint(max_point)
polydata = vtk.vtkPolyData()
polydata.SetPoints(points)

writer_point = vtk.vtkXMLPolyDataWriter()
writer_point.SetFileName("max_point.vtp")
writer_point.SetInputData(polydata)
writer_point.Write()

# === Step 8. Save colored surface ===
writer_surface = vtk.vtkXMLPolyDataWriter()
writer_surface.SetFileName("artery_with_roughness.vtp")
writer_surface.SetInputData(surface)
writer_surface.Write()

print("\nFiles created:")
print("- roughness_results.csv (numerical results)")
print("- max_point.vtp (single point marking hotspot)")
print("- artery_with_roughness.vtp (artery colored by roughness)")

files.download("max_point.vtp")
files.download("artery_with_roughness.vtp")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.1/112.1 MB 6.3 MB/s eta 0:00:00
Please upload your .vtp file (artery surface)


Saving 50degree.vtp to 50degree.vtp
=== Surface Roughness Results ===
Ra: 2.58 µm
Rq: 5.85 µm
Rmax: 116.89 µm
Max roughness at point index 35646, coordinates (mm): (91.90879821777344, 106.43609619140625, 126.33699798583984)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Files created:
- roughness_results.csv (numerical results)
- max_point.vtp (single point marking hotspot)
- artery_with_roughness.vtp (artery colored by roughness)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>